## Imports

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

import math

import optuna
from functools import partial

import pandas as pd
import numpy as np

from torchvision import models
from torchvision import transforms as T
import random
from random import random as rnd

from glob import glob
import os

from sklearn.cluster import KMeans

/root/space-ai/nn-approach/hyperview/hyperenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Constants

In [2]:
SEED = 42
VL_SPLIT  = 0.2

WIDTH = 128
HEIGHT = 128
device = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(device)

print(device)

def set_random_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed) # seed the global NumPy random number generator(RNG)
    torch.manual_seed(seed) # seed the RNG for all devices(both CPU and CUDA) 

set_random_seed(seed = SEED)

base_path = "./data/"

gt_path = base_path + 'train_gt.csv'
wavelength_path = base_path + 'wavelengths.csv'


cuda


## Data imports

In [3]:
class ReduceChannels(nn.Module):
    def __init__(self, in_channels=150, out_channels=3):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)

class GaussianNoise(torch.nn.Module):
    def __init__(self, mean=0.0, std=0.005, p=0.5):
        super().__init__()
        self.std = std
        self.mean = mean
        self.p = p

    def forward(self, img):
        if random.random() < self.p:
            noise = torch.randn_like(img) * self.std + self.mean
            return img + noise
        return img

train_transform = T.Compose([
    ReduceChannels(),  # Resize allinea con Albumentations
    GaussianNoise(std=0.005, p=0.5),  # GaussNoise simulato
    T.RandomRotation(90),  # RandomRotate90
    T.RandomResizedCrop((WIDTH, HEIGHT), scale=(0.95, 1.05), ratio=(0.75, 1.33)),  # RandomResizedCrop
    T.RandomHorizontalFlip(p=0.5),  # Flip orizzontale casuale
    T.RandomVerticalFlip(p=0.5),  # Flip verticale casuale (equivalente a Flip generico)
    T.RandomAffine(degrees=90, translate=(0.05, 0.05)),  # ShiftScaleRotate (senza scaling)
])

eval_transform = T.Compose([
    ReduceChannels(),
    T.Resize((WIDTH, HEIGHT)),
])

In [4]:
gt_df = pd.read_csv(gt_path)
wavelength_df = pd.read_csv(wavelength_path)


def load_data(directory: str, tr = None):
    data = []
    sizes = set()
    all_files = np.array(
        sorted(
            glob(os.path.join(directory, "*.npz")),
            key=lambda x: int(os.path.basename(x).replace(".npz", "")),
        )
    )
    for file_name in all_files:
        with np.load(file_name) as npz:
            
            arr = npz['data']
            mask = npz["mask"]
            
            arr = torch.tensor(arr, dtype=torch.float32)
            mask = torch.tensor(~mask, dtype=torch.float32)
            
            arr = arr * mask
            
            if tr:
                arr = tr(arr)

        sizes.add((arr.shape[1], arr.shape[2]))
        data.append(arr)
    return data, sizes


def load_gt(file_path: str):
    gt_file = pd.read_csv(file_path)
    labels = gt_file[["P", "K", "Mg", "pH"]].values
    return labels

# X_train_grouped, sizes = load_data(base_path + "train_data")
X_train_base, _ = load_data(base_path + "train_data", tr = train_transform)
# X_test_grouped, _ = load_data(base_path + "test_data")
X_test_base, _ = load_data(base_path + "test_data", tr = eval_transform)
y_train_base = load_gt(base_path + "train_gt.csv")

In [5]:
# values = list(sizes)

# kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
# kmeans.fit(values)

# centroids = kmeans.cluster_centers_.astype(int)

# def assegna_gruppo(tupla, centroids):
#     distanze = [np.linalg.norm(tupla - centroide) for centroide in centroids]
#     return np.argmin(distanze)

# transformers = [
#     transforms.Resize(size=(centroids[0][0], centroids[0][1])),
#     transforms.Resize(size=(centroids[1][0], centroids[1][1])),
#     transforms.Resize(size=(centroids[2][0], centroids[2][1]))
# ]

# gruppi = {i: [] for i in range(len(centroids))}
# target_gruppi = {i: [] for i in range(len(centroids))}
# for idx, t in enumerate(X_train_grouped):
#     gruppo = assegna_gruppo(list(t.shape[1:]), centroids)
#     gruppi[gruppo].append(transformers[gruppo](t))
#     target_gruppi[gruppo].append(y_train_base[idx]) 

# X_train_sml = torch.stack(gruppi[1])
# X_train_mid = torch.stack(gruppi[0])
# X_train_big = torch.stack(gruppi[2])

# y_train_sml = torch.tensor(target_gruppi[1], dtype=torch.float32)
# y_train_mid = torch.tensor(target_gruppi[0], dtype=torch.float32)
# y_train_big = torch.tensor(target_gruppi[2], dtype=torch.float32)

# gruppi = {i: [] for i in range(len(centroids))}
# for idx, t in enumerate(X_test_grouped):
#     gruppo = assegna_gruppo(list(t.shape[1:]), centroids)
#     gruppi[gruppo].append(transformers[gruppo](t))
    
# X_test_sml = torch.stack(gruppi[1])
# X_test_mid = torch.stack(gruppi[0])
# X_test_big = torch.stack(gruppi[2])

X_train = torch.stack([x.detach() for x in X_train_base])
X_test = torch.stack([x.detach() for x in X_test_base])
y_train = torch.tensor(y_train_base, dtype=torch.float32)

# print(X_train_sml.shape)
# print(X_train_mid.shape)
# print(X_train_big.shape)

# print(X_test_sml.shape)
# print(X_test_mid.shape)
# print(X_test_big.shape)

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)

torch.Size([1732, 3, 128, 128])
torch.Size([1154, 3, 128, 128])
torch.Size([1732, 4])


In [6]:
dataset=TensorDataset(X_train,y_train)

n = int(len(X_train) * VL_SPLIT)
train_dataset, val_dataset = torch.utils.data.random_split(dataset, [n, len(dataset) - n])

train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=False)

testset=TensorDataset(X_test)

test_dataloader= DataLoader(testset,shuffle=False)

## Training

In [7]:
criterion = nn.MSELoss()

def train_one_epoch(m, o):
    running_loss = 0.
    for data in train_dataloader:
        inputs, labels = data
        o.zero_grad()
        outputs = m(inputs.to(device))
        loss = criterion(outputs, labels.to(device))
        loss.backward(retain_graph=True)
        o.step()
        running_loss += loss.item()
    return running_loss / len(train_dataloader)

In [8]:
def train(m, o, path="", patience=10):
    best_vloss = float('inf')
    patience_counter = 0
    
    for epoch in range(500):
        print(f'============= EPOCH {epoch + 1} =============')
        m.train(True)
        avg_loss = train_one_epoch(m, o)
        
        m.eval()
        running_vloss = 0.0
        with torch.no_grad():
            for vinputs, vlabels in val_dataloader:
                voutputs = m(vinputs.to(device))
                vloss = criterion(voutputs, vlabels.to(device))
                running_vloss += vloss.item()
        
        avg_vloss = running_vloss / len(val_dataloader)
        print(f'LOSS: train {round(avg_loss, 4)} | valid {round(avg_vloss, 4)}')
        
        if avg_vloss < best_vloss:
            best_vloss = avg_vloss
            if path!="":
                torch.save(m.state_dict(), path)
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience or math.isnan(avg_loss) or math.isinf(avg_loss):
            print("Early stopping triggered")
            break

In [9]:
def objective(trial, pretrained=False):
        
    lr = trial.suggest_float("lr", 1e-8, 5e-4)
    momentum = trial.suggest_float("momentum", 0.7, 0.99)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    
    
    if pretrained:
        model = models.swin_v2_b(weights= models.Swin_V2_B_Weights.DEFAULT)

    else:
        model = models.swin_v2_b()
    
    num_features = model.head.in_features
    model.head = torch.nn.Linear(num_features, 4)
    model.to(device)
    
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=momentum, weight_decay=weight_decay)
    
    train(model, optimizer)
    
    model.eval()
    running_vloss = 0.0
    with torch.no_grad():
        for vinputs, vlabels in val_dataloader:
            voutputs = model(vinputs.to(device))
            vloss = criterion(voutputs, vlabels.to(device))
            running_vloss += vloss.item()
    
    return running_vloss / len(val_dataloader)

# Trova i migliori iperparametri
study = optuna.create_study(direction="minimize")
study.optimize(partial(objective, pretrained=True), n_trials=100)


best_params = study.best_params
print("Best hyperparameters:", best_params)

# Riallenamento con i migliori iperparametri
final_model = models.swin_v2_b(weights= models.Swin_V2_B_Weights.DEFAULT)
num_features = final_model.head.in_features
final_model.head = torch.nn.Linear(num_features, 4)
final_model.to(device)
final_optimizer = torch.optim.SGD(final_model.parameters(), lr=best_params["lr"], momentum=best_params["momentum"], weight_decay=best_params["weight_decay"])
train(final_model, final_optimizer, path="best_model.pth")

[I 2025-03-29 16:31:04,956] A new study created in memory with name: no-name-0aed264b-8282-4ee6-bbd4-60bc94659356


============= EPOCH 1 =============
LOSS: train 18440.5508 | valid 1782.9625
============= EPOCH 2 =============
LOSS: train 3010.3089 | valid 2483.325
============= EPOCH 3 =============
LOSS: train 2172.8142 | valid 1826.8356
============= EPOCH 4 =============
LOSS: train 1851.0809 | valid 1679.1318
============= EPOCH 5 =============
LOSS: train 1817.6113 | valid 1668.4903
============= EPOCH 6 =============
LOSS: train 1775.3328 | valid 2099.0578
============= EPOCH 7 =============
LOSS: train 1939.4957 | valid 1576.8895
============= EPOCH 8 =============
LOSS: train 1715.7236 | valid 1718.3444
============= EPOCH 9 =============
LOSS: train 1684.879 | valid 1709.1093
============= EPOCH 10 =============
LOSS: train 1580.5353 | valid 1690.2059
============= EPOCH 11 =============
LOSS: train 1624.8868 | valid 1653.9009
============= EPOCH 12 =============
LOSS: train 1705.7683 | valid 1839.1024
Early stopping triggered


[I 2025-03-29 16:33:06,060] Trial 0 finished with value: 1839.1023781516335 and parameters: {'lr': 0.00011683666277849083, 'momentum': 0.9083633666125726, 'weight_decay': 2.4168195680425517e-06}. Best is trial 0 with value: 1839.1023781516335.


============= EPOCH 1 =============
LOSS: train 16024.5039 | valid 16835.0944
============= EPOCH 2 =============
LOSS: train 13979.2264 | valid 1658.0139
============= EPOCH 3 =============
LOSS: train 3882.5093 | valid 1608.3857
============= EPOCH 4 =============
LOSS: train 2076.1734 | valid 1626.6294
============= EPOCH 5 =============
LOSS: train 1687.0773 | valid 1669.5709
============= EPOCH 6 =============
LOSS: train 1660.4485 | valid 1576.52
============= EPOCH 7 =============
LOSS: train 1653.6574 | valid 1633.2555
============= EPOCH 8 =============
LOSS: train 1643.0204 | valid 1582.6088
============= EPOCH 9 =============
LOSS: train 1567.1505 | valid 1594.0687
============= EPOCH 10 =============
LOSS: train 1547.0082 | valid 1571.9336
============= EPOCH 11 =============
LOSS: train 1658.7051 | valid 1596.8184
============= EPOCH 12 =============
LOSS: train 1845.5772 | valid 1624.1619
============= EPOCH 13 =============
LOSS: train 1711.8988 | valid 1807.6293
=======

[I 2025-03-29 16:35:39,713] Trial 1 finished with value: 1606.1233936656606 and parameters: {'lr': 0.00024782738697392317, 'momentum': 0.7726921422069667, 'weight_decay': 0.0008051289113611174}. Best is trial 1 with value: 1606.1233936656606.


...

============= EPOCH 1 =============
LOSS: train 15633.7694 | valid 2246.7873
============= EPOCH 2 =============
LOSS: train 12610.2528 | valid 9668.4205
============= EPOCH 3 =============
LOSS: train 8212.5848 | valid 3439.2852
============= EPOCH 4 =============
LOSS: train 2871.4269 | valid 2332.1182
============= EPOCH 5 =============
LOSS: train 2726.1073 | valid 3088.7454
============= EPOCH 6 =============
LOSS: train 2764.5 | valid 2009.8097
============= EPOCH 7 =============
LOSS: train 2422.22 | valid 2346.2095
============= EPOCH 8 =============
LOSS: train 1985.6498 | valid 1599.7398
============= EPOCH 9 =============
LOSS: train 1928.6374 | valid 1590.0552
============= EPOCH 10 =============
LOSS: train 1877.9696 | valid 1634.0689
============= EPOCH 11 =============
LOSS: train 1746.1147 | valid 1662.0677
============= EPOCH 12 =============
LOSS: train 1617.4004 | valid 1632.369
============= EPOCH 13 =============
LOSS: train 1886.0529 | valid 1612.4031
============

[I 2025-03-29 16:40:20,980] Trial 3 finished with value: 1596.4351113059304 and parameters: {'lr': 0.00022132394731513163, 'momentum': 0.9100560180559086, 'weight_decay': 5.141115470965403e-05}. Best is trial 3 with value: 1596.4351113059304.


============= EPOCH 1 =============
LOSS: train 15506.6278 | valid 26300.82
============= EPOCH 2 =============
LOSS: train 8545.4106 | valid 10395.0824
============= EPOCH 3 =============
LOSS: train 3051.2739 | valid 2520.8491
============= EPOCH 4 =============
LOSS: train 1938.9575 | valid 1584.6195
============= EPOCH 5 =============
LOSS: train 1749.3235 | valid 1584.6845
============= EPOCH 6 =============
LOSS: train 1582.1957 | valid 1618.1239
============= EPOCH 7 =============
LOSS: train 1626.242 | valid 1578.996
============= EPOCH 8 =============
LOSS: train 1785.7779 | valid 1667.8477
============= EPOCH 9 =============
LOSS: train 1550.3159 | valid 1615.5358
============= EPOCH 10 =============
LOSS: train 1563.6373 | valid 1581.6262
============= EPOCH 11 =============
LOSS: train 1680.8036 | valid 1583.5513
============= EPOCH 12 =============
LOSS: train 1721.05 | valid 1618.7556
Early stopping triggered


[I 2025-03-29 16:42:26,919] Trial 4 finished with value: 1618.7556207830255 and parameters: {'lr': 0.0002509419785429118, 'momentum': 0.8731286682700429, 'weight_decay': 8.491680836564923e-06}. Best is trial 3 with value: 1596.4351113059304.


Best hyperparameters: {'lr': 0.00022132394731513163, 'momentum': 0.9100560180559086, 'weight_decay': 5.141115470965403e-05}
============= EPOCH 1 =============
LOSS: train 16325.4721 | valid 8301.7122
============= EPOCH 2 =============
LOSS: train 4431.5099 | valid 3601.9327
============= EPOCH 3 =============
LOSS: train 2119.3138 | valid 2526.9689
============= EPOCH 4 =============
LOSS: train 1641.1929 | valid 1633.5253
============= EPOCH 5 =============
LOSS: train 1617.0398 | valid 1572.6935
============= EPOCH 6 =============
LOSS: train 1691.8292 | valid 1778.7569
============= EPOCH 7 =============
LOSS: train 1638.5508 | valid 1914.2285
============= EPOCH 8 =============
LOSS: train 1754.2448 | valid 1703.9192
============= EPOCH 9 =============
LOSS: train 1659.9842 | valid 2607.0961
============= EPOCH 10 =============
LOSS: train 1908.3387 | valid 1790.2131
Early stopping triggered


In [10]:
# def predict(model, path):
#     model.eval()
#     predictions = []  # Inizializza una lista per memorizzare le predizioni
#     with torch.no_grad():
#         for _, data in enumerate(test_dataloader):
#             inputs = data[0].to(device)
#             # Effettua le previsioni utilizzando il modello
#             outputs = model(inputs)
#             # Aggiungi le predizioni alla lista delle predizioni
#             predictions.append(outputs.cpu().numpy())

#     predictions_array = np.concatenate(predictions)
#     submission_df = pd.DataFrame(data=predictions_array, columns=["P", "K", "Mg", "pH"])
#     submission_df.to_csv(path, index_label="sample_index")

In [11]:
# predict(pretrained_model, "pretrained_model_submission.csv")
# predict(non_pretrained_model, "non_pretrained_model_submission.csv")